In [ ]:

import torch
import transformers
from transformers import AutoConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is unavailable. Enable a GPU runtime in Colab.")

print("\nGPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("Total GPU memory (GiB):",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

config = AutoConfig.from_pretrained(MODEL_ID)

print("\nModel:", MODEL_ID)
print("Model type:", config.model_type)
print("Layers:", config.num_hidden_layers)
print("Hidden size:", config.hidden_size)
print("Attention heads:", config.num_attention_heads)
print("KV heads:", config.num_key_value_heads)
print("Head dimension:", config.hidden_size // config.num_attention_heads)


PyTorch: 2.11.0+cu130
Transformers: 5.18.0
CUDA available: True

GPU: Tesla T4
Compute capability: (7, 5)
Total GPU memory (GiB): 14.56


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]


Model: Qwen/Qwen2.5-1.5B-Instruct
Model type: qwen2
Layers: 28
Hidden size: 1536
Attention heads: 12
KV heads: 2
Head dimension: 128


In [ ]:
%pip install -q bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.3 MB/s eta 0:00:00


In [ ]:
import gc
import time
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
PROMPT = "Explain how an LLM generates text, step by step."
MAX_NEW_TOKENS = 64

def benchmark(model, label):
    model.eval()
    inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda")

    def generate():
        with torch.inference_mode():
            return model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
            )

    # Warmup
    _ = generate()
    torch.cuda.synchronize()

    torch.cuda.reset_peak_memory_stats()
    start_alloc = torch.cuda.memory_allocated()

    times = []
    output = None
    for _ in range(3):
        torch.cuda.synchronize()
        start = time.perf_counter()
        output = generate()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

    peak = torch.cuda.max_memory_allocated()
    generated = output.shape[1] - inputs["input_ids"].shape[1]

    print(f"\n{label}")
    print("Model footprint (GiB):",
          round(model.get_memory_footprint() / 1024**3, 3))
    print("Peak allocated GPU memory (GiB):",
          round(peak / 1024**3, 3))
    print("Generation latency mean (s):",
          round(sum(times) / len(times), 3))
    print("Generated tokens:", generated)
    print("Output:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

# FP16 baseline: your existing model
benchmark(model, "FP16")

del model
gc.collect()
torch.cuda.empty_cache()

# Load 4-bit quantized model
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

benchmark(model_4bit, "4-bit NF4")

NameError: name 'model' is not defined

In [ ]:

from torch.backends.cuda import (
    is_flash_attention_available,
    flash_sdp_enabled,
    mem_efficient_sdp_enabled,
    math_sdp_enabled,
)

print("Flash attention compiled into PyTorch:",
      is_flash_attention_available())

print("Flash SDP enabled:",
      flash_sdp_enabled())

print("Memory-efficient SDP enabled:",
      mem_efficient_sdp_enabled())

print("Math SDP enabled:",
      math_sdp_enabled())


Flash attention compiled into PyTorch: True
Flash SDP enabled: True
Memory-efficient SDP enabled: True
Math SDP enabled: True


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    attn_implementation="sdpa",
).to("cuda")

model.eval()

print("Requested attention: sdpa")
print("Configured attention:", model.config._attn_implementation)
print("Model device:", model.device)
print("Model dtype:", next(model.parameters()).dtype)
print("GPU memory allocated (GiB):",
      round(torch.cuda.memory_allocated() / 1024**3, 3))

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Requested attention: sdpa
Configured attention: sdpa
Model device: cuda:0
Model dtype: torch.float16
GPU memory allocated (GiB): 2.876


In [ ]:
prompt = "Explain in simple terms why GPUs are useful for LLM inference."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

@torch.inference_mode()
def get_logits():
    return model(**inputs).logits.float()

# First run with SDPA
model.set_attn_implementation("sdpa")
sdpa_logits = get_logits()

# Switch to the standard eager attention implementation
model.set_attn_implementation("eager")
eager_logits = get_logits()

# Compare the same input positions and vocabulary logits
difference = (sdpa_logits - eager_logits).abs()

print("SDPA output shape:", tuple(sdpa_logits.shape))
print("Eager output shape:", tuple(eager_logits.shape))
print("Maximum absolute difference:", difference.max().item())
print("Mean absolute difference:", difference.mean().item())

# Restore SDPA for the next experiment
model.set_attn_implementation("sdpa")

SDPA output shape: (1, 14, 151936)
Eager output shape: (1, 14, 151936)
Maximum absolute difference: nan
Mean absolute difference: nan


In [ ]:
def inspect_logits(name, logits):
    print(f"\n{name}")
    print("  NaN count:", torch.isnan(logits).sum().item())
    print("  +Inf count:", torch.isposinf(logits).sum().item())
    print("  -Inf count:", torch.isneginf(logits).sum().item())
    print("  All finite:", torch.isfinite(logits).all().item())

    finite = logits[torch.isfinite(logits)]
    if finite.numel() > 0:
        print("  Finite min:", finite.min().item())
        print("  Finite max:", finite.max().item())

inspect_logits("SDPA", sdpa_logits)
inspect_logits("Eager", eager_logits)


SDPA
  NaN count: 0
  +Inf count: 0
  -Inf count: 0
  All finite: True
  Finite min: -16.796875
  Finite max: 26.53125

Eager
  NaN count: 2127104
  +Inf count: 0
  -Inf count: 0
  All finite: False


In [ ]:
print("Current model dtype:", next(model.parameters()).dtype)

model.set_attn_implementation("eager")
model.eval()

with torch.inference_mode():
    eager_fp32_logits = model(**inputs).logits.float()

inspect_logits("Eager FP32", eager_fp32_logits)

Current model dtype: torch.float16

Eager FP32
  NaN count: 2127104
  +Inf count: 0
  -Inf count: 0
  All finite: False


In [ ]:
model.set_attn_implementation("sdpa")
model = model.half()
model.eval()

with torch.inference_mode():
    sdpa_fp16_logits = model(**inputs).logits.float()

inspect_logits("SDPA FP16 after restore", sdpa_fp16_logits)


SDPA FP16 after restore
  NaN count: 0
  +Inf count: 0
  -Inf count: 0
  All finite: True
  Finite min: -16.796875
  Finite max: 26.53125


In [ ]:
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

device = "cuda"
dtype = torch.float16

# Representative dimensions: 12 query heads, sequence length 128,
# head dimension 128. This is a kernel test, not the full Qwen model.
q = torch.randn(1, 12, 128, 128, device=device, dtype=dtype)
k = torch.randn(1, 12, 128, 128, device=device, dtype=dtype)
v = torch.randn(1, 12, 128, 128, device=device, dtype=dtype)

backends = {
    "FLASH_ATTENTION": SDPBackend.FLASH_ATTENTION,
    "EFFICIENT_ATTENTION": SDPBackend.EFFICIENT_ATTENTION,
    "MATH": SDPBackend.MATH,
}

for name, backend in backends.items():
    try:
        with sdpa_kernel(backend):
            with torch.inference_mode():
                out = F.scaled_dot_product_attention(q, k, v)
                torch.cuda.synchronize()

        print(
            name,
            "| status: OK",
            "| shape:", tuple(out.shape),
            "| finite:", torch.isfinite(out).all().item(),
        )
    except Exception as e:
        print(name, "| status: FAILED")
        print(type(e).__name__ + ":", str(e)[:500])

FLASH_ATTENTION | status: FAILED
RuntimeError: No available kernel. Aborting execution.
EFFICIENT_ATTENTION | status: OK | shape: (1, 12, 128, 128) | finite: True
MATH | status: OK | shape: (1, 12, 128, 128) | finite: True


/tmp/ipykernel_1332/3304898055.py:24: UserWarning: Memory efficient kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:986.)
  out = F.scaled_dot_product_attention(q, k, v)
/tmp/ipykernel_1332/3304898055.py:24: UserWarning: Memory Efficient attention has been runtime disabled. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/sdp_utils_cpp.h:552.)
  out = F.scaled_dot_product_attention(q, k, v)
/tmp/ipykernel_1332/3304898055.py:24: UserWarning: Flash attention kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:988.)
  out = F.scaled_dot_product_attention(q, k, v)
/tmp/ipykernel_1332/3304898055.py:24: UserWarning: Flash attention only supports gpu architectures in the range [sm80, sm121]. Attempting to run on a sm 7.5 gpu. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:342.)
  out = F.scaled_dot_product_attent

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend
import time

torch.manual_seed(42)

device = "cuda"
dtype = torch.float16

q = torch.randn(1, 12, 128, 128, device=device, dtype=dtype)
k = torch.randn(1, 12, 128, 128, device=device, dtype=dtype)
v = torch.randn(1, 12, 128, 128, device=device, dtype=dtype)

def run_attention(backend):
    with sdpa_kernel(backend):
        with torch.inference_mode():
            return F.scaled_dot_product_attention(q, k, v)

results = {}

for name, backend in [
    ("memory_efficient", SDPBackend.EFFICIENT_ATTENTION),
    ("math", SDPBackend.MATH),
]:
    try:
        # Warmup
        for _ in range(5):
            run_attention(backend)
        torch.cuda.synchronize()

        # Measure repeated calls
        start = time.perf_counter()
        for _ in range(50):
            out = run_attention(backend)
        torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - start) * 1000 / 50

        results[name] = out.float()

        print(
            name,
            "| mean call time:", round(elapsed_ms, 3), "ms",
            "| finite:", torch.isfinite(out).all().item(),
        )
    except Exception as e:
        print(name, "| FAILED:", type(e).__name__, str(e)[:300])

if len(results) == 2:
    diff = (results["memory_efficient"] - results["math"]).abs()
    print("\nMaximum absolute difference:", diff.max().item())
    print("Mean absolute difference:", diff.mean().item())

memory_efficient | mean call time: 0.26 ms | finite: True
math | mean call time: 0.863 ms | finite: True

Maximum absolute difference: 0.00048828125
Mean absolute difference: 1.9400757082621567e-05


In [ ]:
import time
import torch

model.set_attn_implementation("sdpa")
model.eval()

# Create one token sequence, then reuse its tokens to control input length.
base_ids = tokenizer(
    "LLM inference processes prompts and generates tokens using neural networks. ",
    return_tensors="pt",
).input_ids[0]

lengths = [32, 64, 128, 256, 512, 1024]
results = []

for length in lengths:
    token_ids = base_ids.repeat((length + len(base_ids) - 1) // len(base_ids))
    input_ids = token_ids[:length].unsqueeze(0).to("cuda")
    attention_mask = torch.ones_like(input_ids)

    # Warmup
    with torch.inference_mode():
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    torch.cuda.synchronize()

    torch.cuda.reset_peak_memory_stats()
    baseline_mb = torch.cuda.memory_allocated() / 1024**2
    timings = []
    finite = True

    for _ in range(3):
        start = time.perf_counter()
        with torch.inference_mode():
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            )
        torch.cuda.synchronize()
        timings.append((time.perf_counter() - start) * 1000)
        finite = finite and torch.isfinite(output.logits).all().item()
        del output

    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    avg_ms = sum(timings) / len(timings)

    results.append({
        "prompt_tokens": length,
        "mean_prefill_ms": round(avg_ms, 2),
        "peak_memory_mb": round(peak_mb, 1),
        "extra_peak_mb": round(peak_mb - baseline_mb, 1),
        "finite_logits": finite,
    })

    del input_ids, attention_mask
    print(results[-1])

print("\nAll results:")
for row in results:
    print(row)

{'prompt_tokens': 32, 'mean_prefill_ms': 58.69, 'peak_memory_mb': 3030.3, 'extra_peak_mb': 23.2, 'finite_logits': True}
{'prompt_tokens': 64, 'mean_prefill_ms': 62.65, 'peak_memory_mb': 3062.8, 'extra_peak_mb': 47.0, 'finite_logits': True}
{'prompt_tokens': 128, 'mean_prefill_ms': 61.86, 'peak_memory_mb': 3128.0, 'extra_peak_mb': 93.6, 'finite_logits': True}
{'prompt_tokens': 256, 'mean_prefill_ms': 70.67, 'peak_memory_mb': 3258.7, 'extra_peak_mb': 187.3, 'finite_logits': True}
{'prompt_tokens': 512, 'mean_prefill_ms': 154.11, 'peak_memory_mb': 3516.5, 'extra_peak_mb': 370.9, 'finite_logits': True}
{'prompt_tokens': 1024, 'mean_prefill_ms': 371.51, 'peak_memory_mb': 4035.9, 'extra_peak_mb': 741.9, 'finite_logits': True}

All results:
{'prompt_tokens': 32, 'mean_prefill_ms': 58.69, 'peak_memory_mb': 3030.3, 'extra_peak_mb': 23.2, 'finite_logits': True}
{'prompt_tokens': 64, 'mean_prefill_ms': 62.65, 'peak_memory_mb': 3062.8, 'extra_peak_mb': 47.0, 'finite_logits': True}
{'prompt_tokens'